# Sheet 2.4: Training dynamics and scaling

**Roadmap topic 2.4** (Fanout 12.01, 08.04, 12.03). Read first: Chinchilla (Hoffmann et al. 2022) sections 3.1-3.3 and appendix D, Kaplan et al. 2020 sections 1-4, the *Ultra-Scale Playbook* chapters on memory and batch size, Stanford CS336 lectures on scaling.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Exercises 1-3 run on CPU; Exercises 4-5 are Colab training studies with your model from sheet 2.3.

In [ ]:
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import minimize
torch.manual_seed(0)

## Exercise 1: Fit a scaling law and predict a run you didn't do

Chinchilla's parametric form: `L(N, D) = E + A / N^α + B / D^β`.

`fit_scaling_law(N, D, L)` fits `(E, A, α, B, β)` to observed runs. Follow the paper's approach: minimise the **Huber loss** (δ = 1e-3) between `log L̂` and `log L` with L-BFGS, over a grid of initialisations, keeping the best. Parameterise `A = exp(a)`, `B = exp(b)`, `E = exp(e)` for stability (`scipy.optimize.minimize`).

The check generates 24 noisy runs from a hidden law, fits yours, and predicts a held-out larger run.

In [ ]:
def fit_scaling_law(N, D, L):
    # returns dict(E=..., A=..., alpha=..., B=..., beta=...)
    # YOUR CODE
    raise NotImplementedError
def predict(params, N, D):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    true = dict(E=1.69, A=406.4, alpha=0.34, B=410.7, beta=0.28)
    r = np.random.default_rng(0)
    Ns = np.array([2e6, 5e6, 1e7, 2e7, 4e7, 8e7]); Ds = np.array([2e8, 6e8, 2e9, 5e9])
    N, D = np.meshgrid(Ns, Ds); N, D = N.ravel(), D.ravel()
    L = predict(true, N, D) * np.exp(r.normal(0, 0.005, N.shape))
    p = fit_scaling_law(N, D, L)
    held = predict(true, 1.6e8, 1e10); pred = predict(p, 1.6e8, 1e10)
    assert abs(pred - held) / held < 0.02, f"held-out prediction {pred:.3f} vs true {held:.3f}"
    print(f"PASS  fitted alpha={p['alpha']:.3f} beta={p['beta']:.3f} E={p['E']:.3f}; held-out L pred {pred:.3f} vs true {held:.3f}")
_check1()

**Written (1a):** Your fitted α and β won't match the hidden ones exactly, yet the held-out prediction is close. Why? What does that tell you about over-interpreting fitted exponents from small sweeps?

**Your answer:**

## Exercise 2: Compute-optimal allocation

With training compute `C ≈ 6·N·D`, find the `N, D` that minimise `L(N, D)` for a given `C`.
- `optimal_ND(params, C)`: closed form. Derive it: `N_opt = G·(C/6)^(β/(α+β))`, `D_opt = (C/6)/N_opt`, with `G = (αA / (βB))^(1/(α+β))`.
- `tokens_per_param(params, C)` = `D_opt / N_opt`.

In [ ]:
def optimal_ND(p, C):
    # YOUR CODE
    raise NotImplementedError
def tokens_per_param(p, C):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    p = dict(E=1.69, A=406.4, alpha=0.34, B=410.7, beta=0.28)
    for C in (1e18, 1e21, 5.76e23):
        N, D = optimal_ND(p, C)
        assert abs(6 * N * D / C - 1) < 1e-9
        Ns = np.logspace(math.log10(N) - 1, math.log10(N) + 1, 4001)
        Lg = predict(p, Ns, C / 6 / Ns)
        assert abs(Ns[Lg.argmin()] / N - 1) < 0.01, f"C={C:.0e}: not the minimiser"
    N, D = optimal_ND(p, 5.76e23)
    print(f"PASS  Chinchilla budget (5.76e23 FLOPs): N={N/1e9:.0f}B params, D={D/1e12:.2f}T tokens, {D/N:.0f} tokens/param")
_check2()

**Written (2b):** The check prints a compute-optimal model for Chinchilla's own budget using the paper's published parametric fit. Compare it with the model DeepMind actually trained (70B params, 1.4T tokens). Why might they disagree? Read *Chinchilla Scaling: A replication attempt* (Besiroglu et al., 2024) for one answer.

**Written (2a):** Llama-3-8B was trained on ~15T tokens (~1,900 tokens per parameter), far past Chinchilla-optimal. Why is that a rational choice? (Think about who pays for inference.)

**Your answer:**

## Exercise 3: Instrument your training loop

Loss curves alone don't tell you *why* a run spikes. Implement:
- `global_grad_norm(params)`: L2 norm over all gradients.
- `clip_grad_norm(params, max_norm)`: in-place clipping, returns the pre-clip norm (must match `torch.nn.utils.clip_grad_norm_`).
- `UpdateRatioTracker(model)`: call `.before_step()` before `opt.step()` and `.after_step()` after; `.ratios()` returns `{param_name: ‖ΔW‖ / ‖W‖}` for the last step. A healthy LLM run sits around 1e-3; values near 1e-1 warn of instability.
- `max_attention_logit(q, k)`: the largest pre-softmax score `q·kᵀ/√d` in a batch (`q, k (B, H, T, d)`), which you'll log per layer.

In [ ]:
def global_grad_norm(params): raise NotImplementedError
def clip_grad_norm(params, max_norm): raise NotImplementedError
class UpdateRatioTracker:
    def __init__(self, model): raise NotImplementedError
    def before_step(self): raise NotImplementedError
    def after_step(self): raise NotImplementedError
    def ratios(self): raise NotImplementedError
def max_attention_logit(q, k): raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(10, 50), nn.ReLU(), nn.Linear(50, 3)); x = torch.randn(64, 10); y = torch.randint(0, 3, (64,))
    F.cross_entropy(m(x), y).backward()
    m2 = nn.Sequential(nn.Linear(10, 50), nn.ReLU(), nn.Linear(50, 3)); m2.load_state_dict(m.state_dict())
    F.cross_entropy(m2(x), y).backward()
    n1 = clip_grad_norm(list(m.parameters()), 0.1); n2 = torch.nn.utils.clip_grad_norm_(m2.parameters(), 0.1)
    assert torch.allclose(n1, n2), "pre-clip norm differs"
    for a, b in zip(m.parameters(), m2.parameters()): assert torch.allclose(a.grad, b.grad, atol=1e-7), "clipped grads differ"
    opt = torch.optim.SGD(m.parameters(), lr=0.5); tr = UpdateRatioTracker(m)
    w0 = m[0].weight.detach().clone(); tr.before_step(); opt.step(); tr.after_step()
    expect = float((m[0].weight.detach() - w0).norm() / w0.norm())
    assert abs(tr.ratios()["0.weight"] - expect) < 1e-6
    q = torch.randn(2, 3, 8, 16); k = torch.randn(2, 3, 8, 16); k[1, 2, 5] *= 50
    assert abs(max_attention_logit(q, k) - float((q @ k.transpose(-1, -2) / 4).max())) < 1e-5
    print("PASS")
_check3()

## Exercise 4 (Colab T4): batch size vs sequence length at fixed tokens per step

With your 2.3 model (~20M params) on TinyStories, fix **16,384 tokens per optimizer step** and compare `(batch, seq_len)` = (64, 256), (32, 512), (16, 1024). Same LR schedule, same total tokens. Plot validation loss vs tokens and vs wall-clock. Which wins on each axis and why? (Fanout 12.01 covers this.)

## Exercise 5 (Colab T4): cause a loss spike, then fix it two ways

1. Remove warmup and raise peak LR until training spikes or diverges. Log every 10 steps: loss, `global_grad_norm`, per-layer `max_attention_logit`, and update ratios.
2. Find the first signal that moved before the loss did, and which layer it was in.
3. Fix it in two independent ways (e.g. QK-norm, gradient clipping at 1.0, lower β₂, restoring warmup) and show both runs are stable.

Write a one-page incident report: timeline, root cause, fixes, evidence.

## Exercise 6 (Colab, optional): mini scaling law on real runs
Train 5 model sizes × 3 token budgets on TinyStories (smallest runs are minutes each), fit your Exercise 1 law, and predict a 6th, larger run before running it. Report the error.

In [ ]:
# Colab: Exercises 4-6

## Checkpoint
Given only a loss curve and a grad-norm plot, name the likely cause of each pattern: (a) sudden spike that recovers, (b) long plateau early on, (c) divergence to NaN, (d) train loss keeps falling while val loss rises after 1 epoch. Then read the TRM paper (Fanout 12.03, arXiv 2510.04871) and write 5 bullet points on what its recursion changes about the compute/parameter trade-off.

---
### Hints for self-grading
- 1a: A, α, B, β are strongly correlated in the fit; many combinations explain the same data in the sampled range. Predictions inside or near the range are robust; individual exponents are not.
- 2a: Chinchilla minimises training compute for a given loss. When the model is served to many users, a smaller model trained longer costs more to train but much less per inference token.
- Checkpoint: (a) LR too high/bad batch/attention logit growth, (b) LR too low, bad init or a bug (e.g. labels not shifted), (c) LR too high, fp16 overflow, missing clipping, (d) overfitting or data repetition.